# Part 2 — PEFT Methods In A Few-Shot Setting

This notebook is the participant-facing exploration for the PEFT workshop. It keeps the notebook intentionally slim: the reusable experiment code lives in `notebooks/part2_peft_workshop.py`, and plotting helpers live in `notebooks/plot_part3_results.py`.

**Default task:** adapt a fixed pretrained ViT backbone to EuroSAT with 1, 4, and 8 labeled examples per class.

**Methods compared by default:**

- linear probe
- LoRA
- token-space visual prompt tuning
- full fine-tuning baseline

The goal is not to find a universal winner. The goal is to see how method choice changes with data availability, domain gap, overfitting, trainable parameter count, and optimization budget.


In [ ]:
%load_ext autoreload
%autoreload 2

## 0. Colab / Repository Setup

In [ ]:
import os
import sys

if "google.colab" in sys.modules or "COLAB_GPU" in os.environ:
    if not os.path.exists("haicon_peft_co"):
        os.system("git clone https://github.com/trofimova/haicon_peft_co.git")
    os.chdir("haicon_peft_co")
    os.system("pip install -q -r requirements.txt")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


## 1. Imports

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if (ROOT / ".." / "src").exists():
    ROOT = (ROOT / "..").resolve()
elif (ROOT / "src").exists():
    ROOT = ROOT.resolve()

for path in [ROOT, ROOT / "notebooks"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from part2_peft_workshop import WorkshopConfig, get_device, run_experiment_grid
from plot_part3_results import (
    latest_run_dir,
    load_run,
    plot_accuracy_grid,
    plot_generalization_summary,
    plot_shifted_accuracy,
    plot_training_curves,
)

print("repo root:", ROOT)
print("device:", get_device())


## 2. Configuration

This is the main cell to edit. For a live workshop, keep one seed and the default method set. Increase `FULL_FT_EPOCHS` if full fine-tuning looks undertrained.

In [ ]:
DATASET_NAME = "eurosat"
MEDMNIST_NAME = "pathmnist"

METHODS_TO_RUN = ["linear_probe", "lora", "visual_prompt", "full_ft"]
SHOTS_PER_CLASS = [1, 4, 8]
REPEAT_SEEDS = [42]

DEFAULT_EPOCHS = 3
FULL_FT_EPOCHS = 6

config = WorkshopConfig(
    dataset_name=DATASET_NAME,
    medmnist_name=MEDMNIST_NAME,
    data_root=str(ROOT / "data"),
    output_root=str(ROOT / "outputs" / "part3"),
    methods_to_run=METHODS_TO_RUN,
    shots_per_class=SHOTS_PER_CLASS,
    repeat_seeds=REPEAT_SEEDS,
    default_epochs=DEFAULT_EPOCHS,
    epochs_by_method={"full_ft": FULL_FT_EPOCHS},
    batch_size=16,
    eval_per_class=10,
    domain_shift_strength=0.35,
    target_adaptation_shots=0,
    lora_rank=8,
    num_prompt_tokens=8,
    lr_by_method={
        "linear_probe": 5e-3,
        "lora": 3e-3,
        "visual_prompt": 3e-3,
        "full_ft": 1e-5,
    },
)

config


## 3. Run The Experiment Grid

This cell trains each configured method at each shot count. It writes `results.csv`, `results.json`, `history.csv`, and `config.json` to a timestamped folder under `outputs/part3/`.

In [ ]:
run_dir, df, history_df = run_experiment_grid(config)
print("run_dir:", run_dir)

display(
    df[[
        "method", "shots_per_class", "repeat_seed", "epochs",
        "trainable_params", "trainable_pct",
        "train_acc", "source_acc", "target_acc",
        "generalization_gap", "domain_gap",
    ]].sort_values(["shots_per_class", "method"])
)


## 4. In-Domain And Shifted-Domain Accuracy

The first plot evaluates on the usual validation images. The second plot uses the same validation examples after the controlled target-domain distortion.


In [ ]:
df_plot, history_plot, config_plot = load_run(run_dir)

fig = plot_accuracy_grid(df_plot, config_plot)
fig.savefig(run_dir / "accuracy_grid.png", dpi=160)
plt.show()

fig = plot_shifted_accuracy(df_plot, config_plot)
fig.savefig(run_dir / "shifted_accuracy.png", dpi=160)
plt.show()


## 5. Optional Diagnostic Summary

This compact overview is useful for debugging overfitting and domain shift, but it is intentionally secondary to the simpler accuracy and training-curve plots.


In [ ]:
fig = plot_generalization_summary(df_plot)
fig.savefig(run_dir / "generalization_summary.png", dpi=160)
plt.show()


## 6. Training Curves

In [ ]:
fig = plot_training_curves(history_plot)
fig.savefig(run_dir / "training_curves.png", dpi=160)
plt.show()


## 7. Trainable Parameter Counts

In [ ]:
param_df = (
    df_plot.groupby("method", as_index=False)["trainable_params"]
    .first()
    .sort_values("trainable_params")
)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(param_df["method"], param_df["trainable_params"], color="#4c72b0")
ax.set_xscale("log")
ax.set_xlabel("Trainable parameters (log scale)")
ax.set_title("Task-specific trainable parameter count")
for patch, value in zip(ax.patches, param_df["trainable_params"]):
    ax.text(value * 1.05, patch.get_y() + patch.get_height() / 2, f"{value:,}", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(run_dir / "trainable_params.png", dpi=160)
plt.show()


## 8. Replot A Saved Run

Use this after restarting the notebook, or to inspect an older run. Leave `RUN_DIR = None` to use the newest output folder.

In [ ]:
RUN_DIR = None

selected_run_dir = Path(RUN_DIR).resolve() if RUN_DIR is not None else latest_run_dir(ROOT / "outputs" / "part3")
df_plot, history_plot, config_plot = load_run(selected_run_dir)
print("selected run:", selected_run_dir)
display(df_plot.head())

for filename, make_fig in [
    ("accuracy_grid.png", lambda: plot_accuracy_grid(df_plot, config_plot)),
    ("shifted_accuracy.png", lambda: plot_shifted_accuracy(df_plot, config_plot)),
    ("training_curves.png", lambda: plot_training_curves(history_plot)),
    ("generalization_summary.png", lambda: plot_generalization_summary(df_plot)),
]:
    fig = make_fig()
    fig.savefig(selected_run_dir / filename, dpi=160)
    plt.show()


## Optional Extensions

Try one change at a time:

- Switch `DATASET_NAME` from `"eurosat"` to `"cifar10"`.
- Add repeated seeds: `REPEAT_SEEDS = [41, 42, 43]`.
- Increase `FULL_FT_EPOCHS` if full fine-tuning is undertrained.
- Reduce `FULL_FT_EPOCHS` if full fine-tuning overfits too quickly.
- Increase `domain_shift_strength` to make the target split harder.
- Add `target_adaptation_shots=1` to see whether a tiny amount of shifted target data helps.
